# Crowdsourced Wait-Time Prediction: Demo and Rebuild
**All evaluated data is SYNTHETIC.** It comes from a seeded generator whose settings are the author's assumptions, not measurements.

Get the package zip from GitHub (https://github.com/kamrynsschmidt/campus-dining-wait-prediction → **Code → Download ZIP**), then run the cells top to bottom and upload that zip when prompted.

In [ ]:
# 1. Upload and unzip the reproducibility package
#    Pick the ZIP file (any name is fine), not the .ipynb notebook.
import glob, os
from google.colab import files
%cd /content
zips = glob.glob('/content/*.zip')
if not zips:
    up = files.upload()
    zips = [f'/content/{k}' for k in up if k.endswith('.zip')]
assert zips, 'Please upload the reproducibility .zip file'
zip_name = zips[0]
print('Using', zip_name)
!rm -rf /content/repro
!unzip -o -q "{zip_name}" -d /content/repro
# A GitHub 'Download ZIP' puts everything inside one extra folder; step into it if so
root = '/content/repro'
inner = [d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))]
if 'waitsim.py' not in os.listdir(root) and len(inner) == 1:
    root = os.path.join(root, inner[0])
%cd {root}
!pip install -q -r requirements.txt
!ls

## 2. Generator settings (every value is an assumption)

In [ ]:
import json
print(json.dumps(json.load(open('generator_settings.json')), indent=2))

## 3. The real log: two author form tests, excluded from evaluation

In [ ]:
import pandas as pd, waitsim as w
real, log = w.clean(pd.read_csv('wait_log.csv'))
print(log)
real[['line_join_time', 'visual_queue_category', 'supplied_wait', 'wait']]

## 4. Generate one synthetic dataset and clean it
Cleaning recomputes each wait from the two timestamps and rejects reversed or mismatched rows. It should remove exactly the 12 injected bad rows.

In [ ]:
raw = w.generate(w.load_settings(), seed=1)
bad = raw[raw['flag'].astype(str).str.startswith('injected')]
print(bad[['flag', 'line_join_time', 'card_tap_time', 'actual_wait_minutes']].to_string())
df, log = w.clean(raw)
print('\nCleaning log:', log)

## 5. Quick live comparison: 3 baselines vs 3 models, 5-fold CV grouped by date (one dataset, about 30 s)

In [ ]:
w.evaluate(df, grouped=True).round(2)

In [ ]:
print('Shuffled (ungrouped) CV, for comparison:')
w.evaluate(df, grouped=False).round(2)

## 6. Full rebuild: all experiments and figures in the paper (about 10–20 minutes)

In [ ]:
!python run_all.py

## 7. Headline results (compare with Table 2 in the paper)

In [ ]:
pd.read_csv('results/e1_summary.csv', header=[0, 1], index_col=[0, 1])

In [ ]:
pd.read_csv('results/e2_summary.csv')

## 8. Figures

In [ ]:
from IPython.display import Image, display
for f in ['fig_main', 'fig_scatter', 'fig_shift', 'fig_samples']:
    display(Image(f'figures/{f}.png', width=500))